# Milestone 3
This milestone is designed to familiarize you with the core mechanics of Retrieval-Augmented Generation (RAG). You will learn how to query a vector index for relevant context and use a Cross-Encoder to re-rank the results for maximum accuracy. You will then run side-by-side A/B tests to compare a model's zero-shot inference with and without this context. Finally, we will explore the physical limits of context windows (tuning the correct number of chunks to retrieve) and demonstrate the catastrophic dangers of feeding an LLM incorrect data.

In [1]:
# Milestone 3 setup

import pandas as pd
import numpy as np
import faiss #facebook AI Similarity Search
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('../../data/raw/train.csv')

print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating knowledge base
Loading embedding model and creating index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge base successfully created


In [2]:
# Zero-shot classifier for Q1, Q2, Q6
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [25]:
zero_shot_result = zs(prompt_150, labels_150)

predicted_labels = zero_shot_result['labels']
predicted_scores = zero_shot_result['scores']

correct_label_index = predicted_labels.index(ans_150)
round(predicted_scores[correct_label_index],3)

0.384

Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [20]:
query_embedding=model.encode([prompt_150])

# Geting top 10 similar ques from know. base
distances, retrieved_indices = index.search(query_embedding, 10)  # type: ignore
retrieved_indices = retrieved_indices[0]

retrieved_questions = [kb[i] for i in retrieved_indices]
# rank of label_150 in retrieved_questions
retrieved_questions.index(ans_150) + 1 if ans_150 in retrieved_questions else None

10

## Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)
In Question 2, you saw that our FAISS database did not put the true document at rank #1. Why? Because FAISS uses Bi-encoder.

A Bi-encoder embeds the question and the document separately and just compares the distance (Cosine Similarity). They are fast and allows you to search millions of documents but often miss semantic context.

To fix this we use a 2 stage pipeline:
1. Retrieval: Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.
2. Reranking: Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.

A Cross-Encoder passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.

For this we part the prompt and the context and ask it to predict a score.

https://huggingface.co/cross-encoder

In [26]:
# Code to use a Cross-Encoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices] #Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [34]:
ranked_pairs=list(zip(docs_10, ce_scores)) # pairing each context with its score
ranked_pairs.sort(key=lambda x: x[1], reverse=True)

#Rank of ans_150 in the ranked pairs
ranked_pairs.index((ans_150, ce_scores[retrieved_questions.index(ans_150)])) + 1 if ans_150 in retrieved_questions else None

1

Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [45]:
row_42= train.iloc[42]
prompt_42 = str(row_42['prompt'])

query_embedding_42=model.encode([prompt_42])
D, I = index.search(query_embedding_42, 5)  # type: ignore
retrieved_indices_42 = I[0]

concatenated_docs=" ".join([kb[i] for i in retrieved_indices_42])
final_prompt=f"Context: {concatenated_docs} Question: {prompt_42}"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
token=tokenizer.encode(final_prompt, return_tensors="pt", truncation=False)
token.shape[1]

216

Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

In [47]:
true_doc_150=kb[150]
string=f"Context: {true_doc_150} Question: {prompt_150}"
zero_shot_result=zs(string, labels_150)

predicted_labels = zero_shot_result['labels']
predicted_scores = zero_shot_result['scores']

correct_label_index = predicted_labels.index(ans_150)
round(predicted_scores[correct_label_index],3)

0.989

## Concept: The Danger of Bad Retrieval (Adversarial RAG)
The golden rule of Retrieval-Augmented Generation is “Garbage In, Garbage Out.” An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constricting the number of chunks that we give to the model for the same reason.

Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [48]:
wrong_doc=kb[999]
wrong_prompt=f"Context: {wrong_doc} Question: {prompt_150}"

wp_zero_shot_result=zs(wrong_prompt, labels_150)

wp_predicted_labels = wp_zero_shot_result['labels']
wp_predicted_scores = wp_zero_shot_result['scores']

correct_label_index = wp_predicted_labels.index(ans_150)
round(wp_predicted_scores[correct_label_index],3)

0.529

In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [50]:
hits=0
total_rows=100
for i in range(total_rows):
    row= train.iloc[i]
    prompt = str(row['prompt'])
    ans= str(row[row['answer']])

    query_embedding=model.encode([prompt])

    D,I=index.search(query_embedding, 5)  # type: ignore
    retrieved_indices = I[0]
    retrieved_docs=[kb[i] for i in retrieved_indices]

    is_hit = ans in retrieved_docs
    if is_hit:
        hits += 1

hit_rate_percentage = (hits / total_rows) * 100
hit_rate_percentage

73.0

Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [58]:
def average_prediction_score3(prediction:list, ground_truth:str) -> float:
    if not prediction:
        return 0.0
    score = 0.0
    for i, pred in enumerate(prediction[:3]):
        if pred == ground_truth:
            score += 1 / (i + 1)
    return score

total_score = 0.0
for i in range(20):
    row=train.iloc[i]
    prompt=row['prompt']
    labels=[row['A'], row['B'], row['C'], row['D'], row['E']]

    label_map={label:letter for label, letter in zip(labels, ['A', 'B', 'C', 'D', 'E'])}

    # Retrieve
    query_embedding=model.encode([prompt])
    D,I=index.search(query_embedding, 5)  # type: ignore
    retrieved_indices = I[0]
    docs_5=[kb[i] for i in retrieved_indices]

    #Rerank
    pairs=[[prompt, doc] for doc in docs_5]
    ce_scores=cross_encoder.predict(pairs)
    best_doc=docs_5[np.argmax(ce_scores)]

    #Augment
    final_prompt=f"Context: {best_doc} Question: {prompt}"

    #Predict
    zero_shot_result=zs(final_prompt, labels)

    #Scores
    predicted_labels = zero_shot_result['labels']
    predicted_letters=[label_map.get(label, label) for label in predicted_labels]
    top_3=predicted_letters[:3]

    total_score += average_prediction_score3(top_3, row['answer'])

print(f"Total Average Score MAP@3: {total_score / 20}")

Total Average Score MAP@3: 0.975
